%% [markdown]<br>
# Data mining \& clustering

%%

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
import codecs
import re
import os.path
import sklearn
from collections import Counter
from wordcloud import WordCloud, STOPWORDS
import pandas as pd

In [ ]:
from sklearn.metrics import rand_score, adjusted_rand_score
from sklearn.metrics.cluster import contingency_matrix

%% [markdown]<br>
## Data loading

%%

In [ ]:
from sklearn.datasets import fetch_20newsgroups
newsgroups_train = fetch_20newsgroups(subset='train')

%%<br>
conversion BoW + tf-idf

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(max_df=0.95, min_df=2, max_features=1000, stop_words='english')

In [ ]:
vectors = vectorizer.fit_transform(newsgroups_train.data)
print(vectors.shape)

sparsity measure

In [ ]:
print(vectors.nnz / float(vectors.shape[0]))

%%<br>
retrieve words

In [ ]:
print([(i,vectorizer.get_feature_names_out()[i]) \
       for i in np.random.randint(vectors.shape[1], size=10)])

%%<br>
labels (only for evaluation)

In [ ]:
Y = newsgroups_train.target
print(Y[:10])
print([newsgroups_train.target_names[i] for i in Y[:20]]) # vraie classe

%% [markdown]<br>
# 0) Word clouds

%%

In [ ]:
data = np.array(newsgroups_train.data)
corpus = "".join(data)
words = corpus.split()
print("Nb mots=",len(words))

Lets find the most frequent words

In [ ]:
word_counts = Counter(words)
most_common = word_counts.most_common(50)
print("Top 10 frequent words:", most_common[:10])

%% [markdown]<br>
### Plot the N frequent words and verify that its follows a Zipf law

%%

In [ ]:
frequencies =[count for word, count in word_counts.most_common(200)]

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(range(1, 201), frequencies, marker='o', linestyle='none', markersize=4)
plt.title("Word Frequencies - Verifying Zipf's Law")
plt.xlabel("Rank (log scale)")
plt.ylabel("Frequency (log scale)")
plt.xscale('log')
plt.yscale('log')
plt.grid(True)
plt.show()

%% [markdown]<br>
### Experiment word clouds

%%

In [ ]:
wordcloud = WordCloud(background_color='white', stopwords=STOPWORDS, max_words=100).generate(corpus)

In [ ]:
plt.figure(figsize=(8,6))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis("off")
plt.title("Standard WordCloud")
plt.show()

%% [markdown]<br>
### Use word clouds with generate_from_frequencies.

%%<br>
Retrieve the most words frequencies using the fitted TfidfVectorizer

In [ ]:
feature_names = vectorizer.get_feature_names_out()
tfidf_sum = np.asarray(vectors.sum(axis=0)).ravel()
word_freq_dict = dict(zip(feature_names, tfidf_sum))

In [ ]:
wordcloud_freq = WordCloud(background_color='white', max_words=100).generate_from_frequencies(word_freq_dict)

In [ ]:
plt.figure(figsize=(8,6))
plt.imshow(wordcloud_freq, interpolation='bilinear')
plt.axis("off")
plt.title("WordCloud from TF-IDF Frequencies")
plt.show()

%% [markdown]<br>
### Drawing word clouds from classes

%%

In [ ]:
class_idx = 1 # Example: graphics
class_docs =[newsgroups_train.data[i] for i in range(len(Y)) if Y[i] == class_idx]
class_corpus = " ".join(class_docs)

In [ ]:
wordcloud_class = WordCloud(background_color='white', stopwords=STOPWORDS, max_words=100).generate(class_corpus)

In [ ]:
plt.figure(figsize=(8,6))
plt.imshow(wordcloud_class, interpolation='bilinear')
plt.axis("off")
plt.title(f"Word Cloud for Class: {newsgroups_train.target_names[class_idx]}")
plt.show()

%% [markdown]<br>
# 1) Clustering algorithm: K-Means

%%

In [ ]:
from sklearn.cluster import KMeans

In [ ]:
kmeans = KMeans(n_clusters=20, random_state=42, max_iter=10).fit(vectors)

Purity Evaluation function

In [ ]:
def cluster_purity(y_true, y_pred):
    cm = contingency_matrix(y_true, y_pred)
    return np.sum(np.amax(cm, axis=0)) / np.sum(cm)

In [ ]:
purity_km = cluster_purity(Y, kmeans.labels_)
rs_km = rand_score(Y, kmeans.labels_)
ars_km = adjusted_rand_score(Y, kmeans.labels_)

In [ ]:
print("--- K-Means Evaluation ---")
print(f"Purity: {purity_km:.4f}")
print(f"Rand Score: {rs_km:.4f}")
print(f"Adjusted Rand Score: {ars_km:.4f}")

Qualitative: Getting top words for clusters

In [ ]:
order_centroids = kmeans.cluster_centers_.argsort()[:, ::-1]
terms = vectorizer.get_feature_names_out()
print("\nTop words per K-Means cluster:")
for i in range(5): # Showing 5 clusters for brevity
    top_words = [terms[ind] for ind in order_centroids[i, :8]]
    print(f"Cluster {i}: {', '.join(top_words)}")

%% [markdown]<br>
# 2) Latent Semantic Analysis (LSA <=> SVD)

%%

In [ ]:
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer

Running LSA to reduce to 50 dimensions

In [ ]:
svd = TruncatedSVD(n_components=50, random_state=42)
U = svd.fit_transform(vectors)

Normalizing LSA output (L2) before K-Means

In [ ]:
U_normalized = Normalizer(copy=False).fit_transform(U)
vectors_SVDn = U_normalized # For visualization

K-Means over LSA

In [ ]:
kmeans_lsa = KMeans(n_clusters=20, random_state=42, max_iter=10).fit(U_normalized)

In [ ]:
purity_lsa = cluster_purity(Y, kmeans_lsa.labels_)
rs_lsa = rand_score(Y, kmeans_lsa.labels_)
ars_lsa = adjusted_rand_score(Y, kmeans_lsa.labels_)

In [ ]:
print("--- LSA + K-Means Evaluation ---")
print(f"Purity: {purity_lsa:.4f}")
print(f"Rand Score: {rs_lsa:.4f}")
print(f"Adjusted Rand Score: {ars_lsa:.4f}")

%%<br>
t-SNE from the U matrix computed by LSA

In [ ]:
from sklearn.manifold import TSNE
tsne = TSNE(n_components=2, init='pca', n_iter=2000, verbose=0, random_state=42)
tsne_mat = tsne.fit_transform(U)

%%

In [ ]:
NN2cluster = np.argmax(np.abs(vectors_SVDn), axis=0) # 50 points that represent highest activation
import matplotlib.cm as cm

In [ ]:
cmap = plt.cm.get_cmap('tab20', 20)

In [ ]:
plt.figure(figsize=(12, 8))
plt.scatter(tsne_mat[:,0], tsne_mat[:,1], c=Y, cmap=cmap, s=10, alpha=0.7)
plt.scatter(tsne_mat[NN2cluster[:],0], tsne_mat[NN2cluster[:],1], c='black', marker='X', s=150, edgecolors='white', label='Highest activations')
plt.colorbar(ticks=range(20), label='True Classes')
plt.title("t-SNE visualization of LSA document embeddings")
plt.legend()
plt.show()

%% [markdown]<br>
# 3) Latent Dirichlet Allocation (LDA)

%%

In [ ]:
from nltk.tokenize import RegexpTokenizer
from sklearn.feature_extraction.text import CountVectorizer

Initialize regex tokenizer

In [ ]:
tokenizer = RegexpTokenizer(r'\w+')

Vectorize document using Count (mandatory for LDA)

In [ ]:
vectorizer_count = CountVectorizer(lowercase=True,
                        stop_words='english',
                        ngram_range = (1,1),
                        tokenizer = tokenizer.tokenize, max_df=0.95, min_df=2, max_features=1000)

In [ ]:
vectors_count = vectorizer_count.fit_transform(newsgroups_train.data)

%%

In [ ]:
from sklearn.decomposition import LatentDirichletAllocation

LDA Model

In [ ]:
lda = LatentDirichletAllocation(n_components=20, random_state=42, max_iter=10)
lda_docs = lda.fit_transform(vectors_count)

Predict cluster by taking the topic with highest probability

In [ ]:
preds_lda = np.argmax(lda_docs, axis=1)

In [ ]:
purity_lda = cluster_purity(Y, preds_lda)
rs_lda = rand_score(Y, preds_lda)
ars_lda = adjusted_rand_score(Y, preds_lda)

In [ ]:
print("--- LDA Evaluation ---")
print(f"Purity: {purity_lda:.4f}")
print(f"Rand Score: {rs_lda:.4f}")
print(f"Adjusted Rand Score: {ars_lda:.4f}")

In [ ]:
print("\nTop words per LDA Topic:")
feature_names_lda = vectorizer_count.get_feature_names_out()
for topic_idx, topic in enumerate(lda.components_[:5]): # show top 5 topics
    top_words = [feature_names_lda[i] for i in topic.argsort()[:-9:-1]]
    print(f"Topic {topic_idx}: {', '.join(top_words)}")

%%[markdown]<br>
# Performances evaluation<br>
Compare the different approaches wrt three quantitative metrics.

%%<br>
Building comparison table

In [ ]:
results_data = {
    'Clustering Model':['K-Means (BoW TF-IDF)', 'LSA + K-Means', 'LDA'],
    'Purity':[purity_km, purity_lsa, purity_lda],
    'Rand Score': [rs_km, rs_lsa, rs_lda],
    'Adjusted Rand Score': [ars_km, ars_lsa, ars_lda]
}

In [ ]:
df_results = pd.DataFrame(results_data)

Print as a nice table

In [ ]:
print("\n=== FINAL QUANTITATIVE COMPARISON ===")
print(df_results.to_markdown(index=False, floatfmt=".4f"))